# PHASE 2 KOMPONEN 2 — LANGKAH 2: COMMUNITY DETECTION
## Social Penetration & Network Diffusion Framework

**Input:**
- `network_edges.csv` — weighted edge list dari Langkah 1
- `network_nodes.xlsx` — node list dengan account_type yang sudah dilabeli manual

**Output:**
- `network_nodes_with_community.csv` — node list + community assignment
- `community_summary.csv` — ringkasan per komunitas

**Method:** Louvain community detection pada undirected version of graph (standard approach)

## Cell 1: Install & Import

In [1]:
!pip install python-louvain openpyxl -q
!pip install --force-reinstall python-louvain -q

import community.community_louvain as community_louvain
import pandas as pd
import networkx as nx
import numpy as np
from collections import Counter

print("All libraries imported successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 105.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
All libraries imported successfully.


## Cell 2: Load Data & Rebuild Graph

Rebuild directed graph dari edge list, lalu apply account_type dari file xlsx yang sudah dilabeli manual.

In [2]:
# Load edge list
edges_df = pd.read_csv('network_edges.csv')
print(f"Edges loaded: {len(edges_df)}")

# Load node labels (yang sudah di-update manual)
nodes_df = pd.read_excel('network_nodes.xlsx')
print(f"Nodes loaded: {len(nodes_df)}")

# Build directed graph
G_directed = nx.DiGraph()
for _, row in edges_df.iterrows():
    G_directed.add_edge(
        row['source'],
        row['target'],
        weight=row['weight'],
        dominant_type=row['dominant_type'],
        dominant_topic=row['dominant_topic']
    )

# Apply account_type dari xlsx
type_map = dict(zip(nodes_df['username'], nodes_df['account_type']))
for node in G_directed.nodes():
    G_directed.nodes[node]['account_type'] = type_map.get(node, 'unknown')

print(f"\nDirected graph: {G_directed.number_of_nodes()} nodes, {G_directed.number_of_edges()} edges")

# Verifikasi: tidak boleh ada unknown
unknown_count = sum(1 for n in G_directed.nodes() if G_directed.nodes[n].get('account_type') == 'unknown')
print(f"Unknown nodes: {unknown_count}")

Edges loaded: 13994
Nodes loaded: 11426

Directed graph: 11426 nodes, 13994 edges
Unknown nodes: 0


## Cell 3: Convert ke Undirected untuk Louvain

Louvain membutuhkan undirected graph. Edge weight di-merge:
jika A→B (weight 3) dan B→A (weight 1) ada, maka edge A—B = weight 4.

In [3]:
# Convert directed -> undirected, merge weights
G_undirected = G_directed.to_undirected()

# Cek apakah weight ter-merge dengan benar
# (nx.to_undirected mengambil salah satu edge jika ada A->B dan B->A,
#  kita perlu manual merge weights)
G_undirected = nx.Graph()
for u, v, data in G_directed.edges(data=True):
    w = data.get('weight', 1)
    if G_undirected.has_edge(u, v):
        G_undirected[u][v]['weight'] += w
    else:
        G_undirected.add_edge(u, v, weight=w)

# Copy node attributes
for node in G_undirected.nodes():
    if node in G_directed.nodes():
        G_undirected.nodes[node]['account_type'] = G_directed.nodes[node].get('account_type', 'unknown')

print(f"Undirected graph: {G_undirected.number_of_nodes()} nodes, {G_undirected.number_of_edges()} edges")
print(f"(Directed had {G_directed.number_of_edges()} edges — reduction from merging bidirectional edges)")

Undirected graph: 11426 nodes, 13923 edges
(Directed had 13994 edges — reduction from merging bidirectional edges)


## Cell 4: Louvain Community Detection

In [4]:
# Jalankan Louvain dengan resolution=1.0 (default)
partition = community_louvain.best_partition(
    G_undirected,
    weight='weight',
    resolution=1.0,
    random_state=42
)

# Modularity score
modularity = community_louvain.modularity(partition, G_undirected, weight='weight')

num_communities = len(set(partition.values()))

print(f"Modularity: {modularity:.4f}")
print(f"Number of communities: {num_communities}")

Modularity: 0.7991
Number of communities: 1667


## Cell 5: Community Size Distribution

In [5]:
# Hitung ukuran setiap komunitas
community_sizes = Counter(partition.values())
sizes_sorted = sorted(community_sizes.items(), key=lambda x: x[1], reverse=True)

print(f"Total communities: {len(sizes_sorted)}")
print(f"\nTop 15 communities by size:")
print(f"{'Community':<12} {'Size':<8} {'% of network':<15}")
print("-" * 35)
for comm_id, size in sizes_sorted[:15]:
    pct = size / G_undirected.number_of_nodes() * 100
    print(f"{comm_id:<12} {size:<8} {pct:.1f}%")

# Size distribution stats
all_sizes = [s for _, s in sizes_sorted]
print(f"\nSize distribution:")
print(f"  Min: {min(all_sizes)}")
print(f"  Max: {max(all_sizes)}")
print(f"  Mean: {np.mean(all_sizes):.1f}")
print(f"  Median: {np.median(all_sizes):.0f}")
print(f"  Communities with 1 node: {sum(1 for s in all_sizes if s == 1)}")
print(f"  Communities with >= 10 nodes: {sum(1 for s in all_sizes if s >= 10)}")
print(f"  Communities with >= 50 nodes: {sum(1 for s in all_sizes if s >= 50)}")
print(f"  Communities with >= 100 nodes: {sum(1 for s in all_sizes if s >= 100)}")

Total communities: 1667

Top 15 communities by size:
Community    Size     % of network   
-----------------------------------
2            1075     9.4%
7            581      5.1%
9            459      4.0%
12           433      3.8%
11           346      3.0%
3            333      2.9%
10           332      2.9%
86           244      2.1%
1            210      1.8%
43           203      1.8%
6            202      1.8%
70           187      1.6%
58           177      1.5%
49           175      1.5%
69           170      1.5%

Size distribution:
  Min: 2
  Max: 1075
  Mean: 6.9
  Median: 2
  Communities with 1 node: 0
  Communities with >= 10 nodes: 59
  Communities with >= 50 nodes: 30
  Communities with >= 100 nodes: 25


## Cell 6: Community Composition — Account Type per Community

Untuk setiap komunitas besar, lihat komposisi grassroot vs media vs elite.
Ini penting untuk analisis cross-community penetration nanti.

In [6]:
# Buat dataframe node + community
node_comm_df = pd.DataFrame([
    {
        'username': node,
        'community': partition[node],
        'account_type': G_undirected.nodes[node].get('account_type', 'unknown')
    }
    for node in G_undirected.nodes()
])

# Komposisi per komunitas (top 10 komunitas terbesar)
top_communities = [comm_id for comm_id, _ in sizes_sorted[:10]]

print("Account type composition per community (top 10):")
print(f"{'Comm':<6} {'Size':<7} {'Grassroot':<12} {'Media':<12} {'Elite':<12}")
print("-" * 49)

for comm_id in top_communities:
    comm_nodes = node_comm_df[node_comm_df['community'] == comm_id]
    size = len(comm_nodes)
    types = comm_nodes['account_type'].value_counts()

    gr = types.get('grassroot', 0)
    mn = types.get('media/news', 0)
    ep = types.get('elite/political', 0)

    print(f"{comm_id:<6} {size:<7} {gr} ({gr/size*100:.0f}%){'':>3} {mn} ({mn/size*100:.0f}%){'':>3} {ep} ({ep/size*100:.0f}%)")

Account type composition per community (top 10):
Comm   Size    Grassroot    Media        Elite       
-------------------------------------------------
2      1075    1016 (95%)    9 (1%)    50 (5%)
7      581     572 (98%)    8 (1%)    1 (0%)
9      459     439 (96%)    11 (2%)    9 (2%)
12     433     422 (97%)    4 (1%)    7 (2%)
11     346     337 (97%)    7 (2%)    2 (1%)
3      333     326 (98%)    5 (2%)    2 (1%)
10     332     331 (100%)    1 (0%)    0 (0%)
86     244     242 (99%)    0 (0%)    2 (1%)
1      210     206 (98%)    3 (1%)    1 (0%)
43     203     202 (100%)    0 (0%)    1 (0%)


## Cell 7: Key Nodes per Community

Siapa node paling penting di setiap komunitas besar? (by degree dalam komunitas)

In [7]:
# Untuk setiap komunitas besar, tampilkan top 3 node by in-degree (dari directed graph)
print("Top 3 nodes per community (by in-degree in directed graph):")
print("=" * 60)

for comm_id in top_communities[:10]:
    comm_nodes_list = node_comm_df[node_comm_df['community'] == comm_id]['username'].tolist()

    # In-degree dari directed graph, hanya untuk node di komunitas ini
    in_deg = [(n, G_directed.in_degree(n)) for n in comm_nodes_list if n in G_directed.nodes()]
    in_deg_sorted = sorted(in_deg, key=lambda x: x[1], reverse=True)[:3]

    size = len(comm_nodes_list)
    print(f"\nCommunity {comm_id} (size: {size}):")
    for user, deg in in_deg_sorted:
        atype = G_directed.nodes[user].get('account_type', '?')
        print(f"  {user}: in_degree={deg} ({atype})")

Top 3 nodes per community (by in-degree in directed graph):

Community 2 (size: 1075):
  prabowo: in_degree=621 (elite/political)
  dpr_ri: in_degree=483 (elite/political)
  listyosigitp: in_degree=113 (grassroot)

Community 7 (size: 581):
  barengwarga: in_degree=261 (elite/political)
  budibukanintel: in_degree=127 (grassroot)
  synthreb3l: in_degree=20 (grassroot)

Community 9 (size: 459):
  kompascom: in_degree=195 (media/news)
  bospurwa: in_degree=72 (grassroot)
  kompastv: in_degree=39 (media/news)

Community 12 (size: 433):
  divhumas_polri: in_degree=160 (elite/political)
  jateng_twit: in_degree=77 (grassroot)
  radioelshinta: in_degree=57 (media/news)

Community 11 (size: 346):
  mdy_asmara1701: in_degree=85 (grassroot)
  neveral0nely___: in_degree=27 (grassroot)
  ardisatriawan: in_degree=25 (grassroot)

Community 3 (size: 333):
  arsipaja: in_degree=153 (media/news)
  kangmanto123: in_degree=49 (grassroot)
  tirtoid: in_degree=44 (media/news)

Community 10 (size: 332):
  t

## Cell 8: Cross-Community Edges

Berapa banyak edge yang menghubungkan node di komunitas berbeda?
Ini adalah dasar untuk mengukur cross-community penetration di Langkah 3.

In [8]:
# Hitung intra-community vs inter-community edges (dari directed graph)
intra_edges = 0
inter_edges = 0

for u, v in G_directed.edges():
    comm_u = partition.get(u)
    comm_v = partition.get(v)
    if comm_u is not None and comm_v is not None:
        if comm_u == comm_v:
            intra_edges += 1
        else:
            inter_edges += 1

total = intra_edges + inter_edges
print(f"Intra-community edges: {intra_edges} ({intra_edges/total*100:.1f}%)")
print(f"Inter-community edges: {inter_edges} ({inter_edges/total*100:.1f}%)")
print(f"Total: {total}")

# Inter-community edge matrix (top communities)
print(f"\nInter-community connections between top 5 communities:")
top5 = top_communities[:5]
print(f"{'':>8}", end='')
for c in top5:
    print(f"  C{c:<5}", end='')
print()

for c1 in top5:
    print(f"C{c1:<6}", end='')
    nodes_c1 = set(node_comm_df[node_comm_df['community'] == c1]['username'])
    for c2 in top5:
        nodes_c2 = set(node_comm_df[node_comm_df['community'] == c2]['username'])
        count = sum(1 for u, v in G_directed.edges() if u in nodes_c1 and v in nodes_c2)
        print(f"  {count:<6}", end='')
    print()

Intra-community edges: 11751 (84.0%)
Inter-community edges: 2243 (16.0%)
Total: 13994

Inter-community connections between top 5 communities:
          C2      C7      C9      C12     C11   
C2       1748    16      80      55      27    
C7       27      640     6       9       10    
C9       95      5       511     11      14    
C12      62      6       8       475     6     
C11      53      9       16      6       378   


## Cell 9: Save Outputs

In [9]:
# 1. Node list + community assignment
node_output = node_comm_df.copy()

# Tambahkan degree info dari directed graph
node_output['in_degree'] = node_output['username'].apply(
    lambda x: G_directed.in_degree(x) if x in G_directed.nodes() else 0
)
node_output['out_degree'] = node_output['username'].apply(
    lambda x: G_directed.out_degree(x) if x in G_directed.nodes() else 0
)

node_output.to_csv('network_nodes_with_community.csv', index=False)

# 2. Community summary
comm_summary = []
for comm_id, size in sizes_sorted:
    comm_nodes = node_comm_df[node_comm_df['community'] == comm_id]
    types = comm_nodes['account_type'].value_counts()

    # Top node by in-degree
    comm_list = comm_nodes['username'].tolist()
    in_deg = [(n, G_directed.in_degree(n)) for n in comm_list if n in G_directed.nodes()]
    top_node = sorted(in_deg, key=lambda x: x[1], reverse=True)[0] if in_deg else ('', 0)

    comm_summary.append({
        'community': comm_id,
        'size': size,
        'grassroot': types.get('grassroot', 0),
        'media_news': types.get('media/news', 0),
        'elite_political': types.get('elite/political', 0),
        'top_node': top_node[0],
        'top_node_in_degree': top_node[1]
    })

comm_summary_df = pd.DataFrame(comm_summary)
comm_summary_df.to_csv('community_summary.csv', index=False)

print(f"Saved: network_nodes_with_community.csv ({len(node_output)} nodes)")
print(f"Saved: community_summary.csv ({len(comm_summary_df)} communities)")
print(f"\nModularity: {modularity:.4f}")
print(f"\n--- Langkah 2 selesai. Ready untuk Langkah 3 (Social Penetration Metrics). ---")

Saved: network_nodes_with_community.csv (11426 nodes)
Saved: community_summary.csv (1667 communities)

Modularity: 0.7991

--- Langkah 2 selesai. Ready untuk Langkah 3 (Social Penetration Metrics). ---
